# 🏥 VietMedKG — Direct Vector Ingestion vào Qdrant Cloud

Notebook này **không cần Ollama, không cần LLM**.

Pipeline:
```
preprocessed_data.csv
    → chuyển mỗi bệnh thành đoạn văn tiếng Việt
    → embed bằng BAAI/bge-m3 (sentence-transformers, GPU T4)
    → upsert thẳng vào Qdrant Cloud (collection: lightrag_vdb_chunks)
```

## ⚙️ Trước khi chạy:
1. **Upload CSV**: Thêm `preprocessed_data.csv` vào Kaggle Dataset (Add Data → New Dataset)
2. **Đặt Secrets** (Add-ons → Secrets):
   - `QDRANT_URL` = `https://dacb4228-6c2b-438f-98c7-3378b52d5b05.eu-west-2-0.aws.cloud.qdrant.io`
   - `QDRANT_API_KEY` = `<your-api-key>`
3. **Bật GPU**: Settings → Accelerator → **GPU T4 x1**

Ước tính: ~5-10 phút cho 8806 bệnh (GPU), ~30-40 phút (CPU only).

In [ ]:
# ── Cell 1: Cài đặt dependencies ─────────────────────────────────────────
import subprocess, sys

packages = [
    "qdrant-client",
    "sentence-transformers",
    "pandas",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Cấu hình credentials ─────────────────────────────────────────
import os

# Nếu dùng Kaggle Secrets (khuyến nghị, bảo mật hơn):
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    QDRANT_URL     = secrets.get_secret("QDRANT_URL")
    QDRANT_API_KEY = secrets.get_secret("QDRANT_API_KEY")
    print("✅ Credentials loaded from Kaggle Secrets")
except Exception:
    # Fallback: điền trực tiếp (không khuyến nghị nếu notebook public)
    QDRANT_URL     = "https://dacb4228-6c2b-438f-98c7-3378b52d5b05.eu-west-2-0.aws.cloud.qdrant.io"
    QDRANT_API_KEY = "YOUR_API_KEY_HERE"   # ← thay bằng API key thật
    print("⚠️  Credentials hardcoded — hãy dùng Kaggle Secrets cho môi trường thật")

# Tên collection trên Qdrant — phải khớp với LightRAG service
COLLECTION_NAME = "lightrag_vdb_chunks"
EMBEDDING_DIM   = 1024
WORKSPACE_ID    = "_"          # DEFAULT_WORKSPACE của LightRAG
BATCH_SIZE      = 128          # Số bệnh mỗi lần upsert (GPU: 128-256 ổn)
CSV_PATH        = "/kaggle/input/vietmedkg/preprocessed_data.csv"

print(f"Collection : {COLLECTION_NAME}")
print(f"Qdrant URL : {QDRANT_URL[:50]}...")
print(f"Batch size : {BATCH_SIZE}")
print(f"CSV path   : {CSV_PATH}")

In [ ]:
# ── Cell 3: Đọc CSV → chuyển thành đoạn văn tiếng Việt ─────────────────
import pandas as pd

def _safe(val) -> str:
    if val is None or (isinstance(val, float) and val != val):
        return ""
    return str(val).strip()

def row_to_text(row) -> str:
    """Chuyển một dòng CSV thành đoạn văn mô tả bệnh (cùng format với local script)."""
    name = _safe(row.get("disease_name"))
    if not name:
        return ""

    parts = [f"Bệnh: {name}."]

    field_map = [
        ("disease_description", "Mô tả"),
        ("disease_category",    "Loại bệnh"),
        ("disease_cause",       "Nguyên nhân gây bệnh"),
        ("disease_symptom",     "Triệu chứng"),
        ("check_method",        "Phương pháp kiểm tra"),
        ("people_easy_get",     "Đối tượng dễ mắc bệnh"),
        ("associated_disease",  "Bệnh liên quan"),
        ("cure_method",         "Phương pháp điều trị"),
        ("cure_department",     "Khoa điều trị"),
        ("cure_probability",    "Tỉ lệ chữa khỏi"),
        ("drug_recommend",      "Thuốc đề xuất"),
        ("drug_common",         "Thuốc phổ biến"),
        ("drug_detail",         "Thông tin thuốc"),
        ("nutrition_do_eat",    "Nên ăn"),
        ("nutrition_not_eat",   "Không nên ăn"),
        ("nutrition_recommend_eat", "Thực phẩm khuyến nghị"),
        ("disease_prevention",  "Cách phòng tránh"),
    ]

    for col, label in field_map:
        val = _safe(row.get(col))
        if val:
            parts.append(f"{label}: {val}.")

    return "\n".join(parts)


df = pd.read_csv(CSV_PATH, encoding="utf-8")
print(f"📂 CSV loaded: {len(df)} rows, columns: {list(df.columns[:5])}...")

chunks = []
for _, row in df.iterrows():
    text = row_to_text(row)
    if text:
        chunks.append({
            "disease_name": _safe(row.get("disease_name", "")),
            "text": text,
        })

print(f"✅ Generated {len(chunks)} disease chunks")
print("\n--- Sample chunk ---")
print(chunks[0]["text"][:300])

In [ ]:
# ── Cell 4: Helper functions — tạo ID theo đúng format LightRAG/Qdrant ──
import hashlib
import uuid
import time

def make_chunk_id(text: str) -> str:
    """Tạo chunk ID bằng MD5 (khớp với ingest_vectors_direct.py)."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def make_qdrant_point_id(chunk_id: str, workspace: str = "_") -> str:
    """
    Tạo Qdrant point ID theo công thức của LightRAG:
        SHA256(workspace + chunk_id)[:16] → UUID hex (simple format)
    """
    hashed = hashlib.sha256((workspace + chunk_id).encode("utf-8")).digest()
    generated_uuid = uuid.UUID(bytes=hashed[:16], version=4)
    return generated_uuid.hex

# Verify
test_id = make_chunk_id("test")
test_qdrant_id = make_qdrant_point_id(test_id)
print(f"Chunk ID (MD5):     {test_id}")
print(f"Qdrant point ID:    {test_qdrant_id}")
print("✅ ID functions OK")

In [ ]:
# ── Cell 5: Load mô hình bge-m3 từ HuggingFace ───────────────────────────
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

print("⏳ Loading BAAI/bge-m3 (lần đầu ~2-3 phút để tải)...")
model = SentenceTransformer("BAAI/bge-m3", device=device)
print(f"✅ Model loaded | Embedding dim: {model.get_sentence_embedding_dimension()}")

# Smoke test
test_emb = model.encode(["Bệnh sốt xuất huyết"], normalize_embeddings=True)
print(f"✅ Test embedding shape: {test_emb.shape}")

In [ ]:
# ── Cell 6: Kết nối Qdrant và tạo collection nếu chưa có ─────────────────
from qdrant_client import QdrantClient, models

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Kiểm tra collection đã tồn tại chưa
exists = client.collection_exists(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' exists: {exists}")

if not exists:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIM,
            distance=models.Distance.COSINE,
        ),
        hnsw_config=models.HnswConfigDiff(payload_m=16, m=0),
    )
    # Tạo index cho workspace_id (LightRAG cần)
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="workspace_id",
        field_schema=models.KeywordIndexParams(
            type=models.KeywordIndexType.KEYWORD,
            is_tenant=True,
        ),
    )
    print(f"✅ Collection '{COLLECTION_NAME}' created")
else:
    # Đảm bảo index đã có
    try:
        client.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name="workspace_id",
            field_schema=models.KeywordIndexParams(
                type=models.KeywordIndexType.KEYWORD,
                is_tenant=True,
            ),
        )
    except Exception:
        pass  # Index đã tồn tại

# Đếm số chunks hiện có
existing_count = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=models.Filter(
        must=[models.FieldCondition(
            key="workspace_id",
            match=models.MatchValue(value=WORKSPACE_ID),
        )]
    ),
    exact=True,
).count
print(f"📊 Existing chunks in Qdrant: {existing_count} / {len(chunks)}")

In [ ]:
# ── Cell 7: Chạy ingestion — Embed + Upsert vào Qdrant ───────────────────
import numpy as np
from tqdm.auto import tqdm

timestamp = int(time.time())

# Tính sẵn chunk_id cho tất cả chunks
for c in chunks:
    c["chunk_id"] = make_chunk_id(c["text"])
    c["qdrant_id"] = make_qdrant_point_id(c["chunk_id"], WORKSPACE_ID)

print(f"🚀 Starting ingestion: {len(chunks)} chunks | batch_size={BATCH_SIZE}")

success_count = 0
skip_count    = 0
error_count   = 0

total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_num, i in enumerate(tqdm(range(0, len(chunks), BATCH_SIZE), total=total_batches, desc="Batches"), 1):
    batch = chunks[i : i + BATCH_SIZE]
    texts = [c["text"] for c in batch]

    try:
        # Embed với bge-m3 (GPU nếu có)
        embeddings = model.encode(
            texts,
            normalize_embeddings=True,   # cosine similarity cần normalize
            show_progress_bar=False,
            batch_size=32,
        )

        # Build Qdrant points — PHẢI khớp với format LightRAG
        points = [
            models.PointStruct(
                id=c["qdrant_id"],
                vector=embeddings[j].tolist(),
                payload={
                    "id":           c["chunk_id"],      # LightRAG ID_FIELD
                    "workspace_id": WORKSPACE_ID,        # LightRAG WORKSPACE_ID_FIELD
                    "created_at":   timestamp,            # LightRAG CREATED_AT_FIELD
                    "content":      c["text"],            # meta_field: content
                    "full_doc_id":  c["disease_name"] or c["chunk_id"],  # meta_field
                    "file_path":    "preprocessed_data.csv",             # meta_field
                },
            )
            for j, c in enumerate(batch)
        ]

        client.upsert(
            collection_name=COLLECTION_NAME,
            points=points,
            wait=True,
        )
        success_count += len(batch)

    except Exception as e:
        error_count += len(batch)
        print(f"\n❌ Batch {batch_num} failed: {e}")

print(f"\n{'='*60}")
print(f"✅ Success : {success_count:,} chunks")
print(f"❌ Failed  : {error_count:,} chunks")
print(f"{'='*60}")

In [ ]:
# ── Cell 8: Xác minh kết quả ──────────────────────────────────────────────
final_count = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=models.Filter(
        must=[models.FieldCondition(
            key="workspace_id",
            match=models.MatchValue(value=WORKSPACE_ID),
        )]
    ),
    exact=True,
).count

print(f"📊 Final count in Qdrant: {final_count:,} chunks")
print(f"📊 Expected:              {len(chunks):,} chunks")

if final_count >= len(chunks):
    print("✅ INGESTION COMPLETE! Tất cả chunks đã được nạp vào Qdrant.")
else:
    missing = len(chunks) - final_count
    print(f"⚠️  Còn thiếu {missing} chunks — hãy chạy lại Cell 7.")

# Test semantic search thử
print("\n--- Test semantic search ---")
test_query = "bệnh sốt xuất huyết triệu chứng"
query_emb = model.encode([test_query], normalize_embeddings=True)[0]

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_emb.tolist(),
    limit=3,
    with_payload=True,
    score_threshold=0.2,
    query_filter=models.Filter(
        must=[models.FieldCondition(
            key="workspace_id",
            match=models.MatchValue(value=WORKSPACE_ID),
        )]
    ),
).points

print(f"Query: '{test_query}'")
print(f"Top {len(results)} results:")
for r in results:
    print(f"  Score={r.score:.3f} | Disease={r.payload.get('full_doc_id', 'N/A')}")